# Model Tuning

1. AWS settting have to be set correctly they should match the keys defined in the config.env file

In [1]:
import os
import numpy as np
import pandas as pd 
import itertools
import mlflow
from keras.optimizers import Adam
from IPython.display import clear_output

import matplotlib.pyplot as plt

from keras.callbacks import EarlyStopping

from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

from sklearn.model_selection import ParameterGrid
from sklearn.metrics import explained_variance_score, mean_absolute_error, r2_score, mean_squared_error

# custom functions for feature engineering
from helper_functions import * 

from keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

2024-02-07 16:54:50.492558: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-02-07 16:54:50.492589: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-02-07 16:54:50.492628: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-02-07 16:54:50.621237: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Your GPU may run slowly with dtype policy mixed_float16 because it does not have compute capability of at least 7.0. Your GPU:
  NVIDIA GeForce GTX 1060 6GB, compute capability 6.1
See https://developer.nvidia.com/cuda-gpus for a list of GPUs and their compute capabilities.
If you will use compatible GPU(s) not attached to this host, e.g. by running a multi-worker model, you can ignore this warning. This message will only be logged once


2024-02-07 16:54:54.701969: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-02-07 16:54:54.836295: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-02-07 16:54:54.836469: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

In [2]:
# setting up credentials for storing the model
os.environ["AWS_ACCESS_KEY_ID"] = "eITEO5kyE7hccuy7UTHv"
os.environ["AWS_SECRET_ACCESS_KEY"] = "5KBCscit30Z70bSVGIMvMBBoqV8ydn232o2MW9RA"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = f"http://172.1.0.12:2000"

mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

In [3]:
def create_lstm_model(layers_config):
    model = Sequential()

    for layer_conf in layers_config:
        layer_type = layer_conf["type"]

        if layer_type == "LSTM":
            lstm_kwargs = {
                "units": layer_conf["units"],
                "return_sequences": layer_conf["return_sequences"]
            }
            # Add input_shape only for the first LSTM layer if specified
            if "input_shape" in layer_conf:
                lstm_kwargs["input_shape"] = layer_conf["input_shape"]

            model.add(LSTM(**lstm_kwargs))

        elif layer_type == "Dropout":
            model.add(Dropout(layer_conf["rate"]))

        elif layer_type == "Dense":
            model.add(Dense(units=layer_conf["units"], activation=layer_conf["activation"]))

    return model

In [4]:
# This code segment could be used to tune a wide variety of feature combinations
# The data preparation phase migth have to be adjusted to make sure sma_6_close_price, etc is avaliable

fixed_features = ['hour_cos', 'hour_sin', 'day_of_week_cos', 'day_of_week_sin',
                   'month_cos', 'month_sin', 'day_of_month_sin', 'day_of_month_cos',
                     'year_normalized', 'is_holiday', 'volume', 'is_weekend', 'count', 'close_price']
optional_features = ['close_lag12','close_lag6', 'close_lag168', 'sma_6_close_price',
                      'ema_6_close_price', 'ema_12_close_price' ]

# generating all combinations of optional features
all_feature_combinations = []
for r in range(len(optional_features) + 1):
    for subset in itertools.combinations(optional_features, r):
        all_feature_combinations.append(fixed_features + list(subset))

print(len(all_feature_combinations))

64


In [5]:
features = ['is_holiday', 'close_price', 'volume', 'is_weekend', 'count', 'hour_cos',
            'hour_sin', 'day_of_week_cos', 'day_of_week_sin', 'month_cos', 'month_sin',
            'day_of_month_sin', 'day_of_month_cos', 'year_normalized']

features_current = ['is_holiday', 'close_price', 'volume', 'is_weekend', 'count', 
                    'hour_cos', 'hour_sin', 'day_of_week_cos', 'day_of_week_sin', 'month_cos', 'month_sin', 'day_of_month_sin', 
                    'day_of_month_cos', 'year_normalized', 'close_lag12', 'close_lag96', 'ema_12_close_price', 'ema_48_close_price', 'ema_96_close_price', 'ema_168_close_price'] 

# all_feature_combinations could be added here in the features entry
param_grid = {
    'features': [features_current],
    'lookback': [12],
    'learning_rate': [0.001],
    'optimizer': ['adam'],
    'batch_size': [16],
    'datatype': [np.float32],
    'epochs': [75],
    'loss': ['mean_absolute_percentage_error'],
    'metrics' : [['mean_absolute_error', 'mean_squared_error', 'accuracy']] # double list to not alternate between the metrics
}

grid = ParameterGrid(param_grid)

len(grid)

1

In [6]:
def data_prep(data):
    data = data.drop(columns=['open_price', 'high', 'low'], axis=1)
    data = process_timestamp(data, fill=True)
    data = add_feature_date(data)
    data = add_holiday_feature(data)
    data = apply_cyclic_encoding(data, columms=['hour', 'day_of_week', 'month'])
    data = apply_day_of_month_encoding(data)
    data = normalize_year(data, columns=['year']) # has to be updated yearly (because of the min-max scaler - max+1 is currently set = 2024)
    data['close_price_true'] = data['close_price'] # save the true price --> for unseen data plot
    data = apply_log_scaler(data, columns=['close_price', 'volume', 'count'])
    data = add_feature_lag(data, lags=[1,4,6,12,24,48,96,168])
    data = simple_moving_average(data, window_sizes=[6,12,24,48,96,168], columns=['close_price']) # window sizes are in hours
    data = exponential_moving_average(data, span_sizes=[6,12,24,48,96,168], columns=['close_price']) # span sizes are in hours
    # not needed as these values are encoded to be cyclic 
    data = data.drop(columns=['symbol_id', 'hour', 'day_of_week', 'day_of_month', 'month', 'year'], axis=1)
    # dropping nan values which are created because of simple_moving_average and lags 
    data = data.dropna()
    return data

In [7]:
# Prep data once for all models: 
file_path = 'ETHUSD.csv'
raw_data = pd.read_csv(file_path)
data = data_prep(raw_data)

test = data.iloc[int(len(data)*0.99):] 
training = data.iloc[:int(len(data)*0.99)]
training.head()

,bucket,close_price,volume,count,is_weekend,is_holiday,hour_cos,hour_sin,day_of_week_cos,day_of_week_sin,...,sma_24_close_price,sma_48_close_price,sma_96_close_price,sma_168_close_price,ema_6_close_price,ema_12_close_price,ema_24_close_price,ema_48_close_price,ema_96_close_price,ema_168_close_price
168,2020-01-08 00:00:00+00:00,5.001595,9.191388,6.827629,0,0,1.000000,0.000000,-0.222521,0.974928,...,4.968375,4.961191,4.936064,4.910448,4.978015,4.972375,4.967661,4.958317,4.942300,4.930113
169,2020-01-08 01:00:00+00:00,4.988867,8.258224,6.230481,0,0,0.965926,0.258819,-0.222521,0.974928,...,4.968918,4.962473,4.937021,4.911123,4.981115,4.974912,4.969357,4.959565,4.943289,4.930914
170,2020-01-08 02:00:00+00:00,4.981069,6.810474,5.332719,0,0,0.866025,0.500000,-0.222521,0.974928,...,4.969082,4.963289,4.937874,4.911732,4.981102,4.975859,4.970294,4.960443,4.944090,4.931597
171,2020-01-08 03:00:00+00:00,4.981824,7.845819,5.480639,0,0,0.707107,0.707107,-0.222521,0.974928,...,4.969406,4.964077,4.938734,4.912384,4.981308,4.976777,4.971217,4.961316,4.944891,4.932280
172,2020-01-08 04:00:00+00:00,4.985386,6.575477,4.709530,0,0,0.500000,0.866025,-0.222521,0.974928,...,4.970174,4.964933,4.939590,4.913055,4.982473,4.978101,4.972350,4.962300,4.945749,4.933000


In [8]:
training.describe()

,close_price,volume,count,is_weekend,is_holiday,hour_cos,hour_sin,day_of_week_cos,day_of_week_sin,month_cos,...,sma_24_close_price,sma_48_close_price,sma_96_close_price,sma_168_close_price,ema_6_close_price,ema_12_close_price,ema_24_close_price,ema_48_close_price,ema_96_close_price,ema_168_close_price
count,32361.000000,32361.000000,32361.000000,32361.000000,32361.000000,3.236100e+04,3.236100e+04,32361.000000,32361.000000,3.236100e+04,...,32361.000000,32361.000000,32361.000000,32361.000000,32361.000000,32361.000000,32361.000000,32361.000000,32361.000000,32361.000000
mean,7.097489,7.144228,6.482961,0.285807,0.034115,1.093618e-04,1.894202e-04,-0.001493,-0.000217,-6.044835e-02,...,7.096624,7.095721,7.093896,7.091137,7.097301,7.097076,7.096624,7.095716,7.093888,7.091143
std,0.966089,1.234987,0.937917,0.451805,0.181528,7.071028e-01,7.071326e-01,0.706949,0.707285,7.006035e-01,...,0.966730,0.967403,0.968787,0.970912,0.966201,0.966347,0.966640,0.967230,0.968419,0.970171
min,4.594008,0.000000,0.000000,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-0.900969,-0.974928,-1.000000e+00,...,4.720352,4.746493,4.766958,4.802116,4.685712,4.716259,4.734013,4.761160,4.789996,4.858750
25%,6.387384,6.371713,5.834811,0.000000,0.000000,-7.071068e-01,-7.071068e-01,-0.900969,-0.781831,-8.660254e-01,...,6.387555,6.387806,6.372224,6.358186,6.386533,6.387257,6.387988,6.384803,6.368067,6.362065
50%,7.414591,7.191399,6.478510,0.000000,0.000000,6.123234e-17,1.224647e-16,-0.222521,0.000000,-1.836970e-16,...,7.413003,7.412882,7.412496,7.414994,7.414091,7.413055,7.413306,7.411747,7.412205,7.412423
75%,7.769188,7.995477,7.100852,1.000000,0.000000,7.071068e-01,7.071068e-01,0.623490,0.781831,5.000000e-01,...,7.768278,7.773128,7.771502,7.778370,7.768661,7.768482,7.770206,7.778063,7.773617,7.771558
max,8.486373,11.697442,10.134480,1.000000,1.000000,1.000000e+00,1.000000e+00,1.000000,0.974928,1.000000e+00,...,8.475145,8.470932,8.464794,8.455181,8.479816,8.476709,8.472832,8.466804,8.457448,8.446745


In [9]:
# Start an MLflow run for each set of parameters
run_count = 0
early_stopping = EarlyStopping(monitor='loss', min_delta=0.01, patience=10, verbose=1, mode='min')

for params in grid:
    with mlflow.start_run():
        clear_output(wait=True)
        print(run_count) 
        # log the currently used parameters
        mlflow.log_params(params)

        # logging training size 
        mlflow.log_param("training data size", len(training))
        mlflow.log_param("unseen test data size", len(test))

        lookback = params['lookback']
        features = params['features']
        n_features = len(features)           

        model_layers_config = [
            {'type': 'LSTM', 'units': n_features * lookback, 'return_sequences': True, 'input_shape': (lookback, n_features)},  
            {'type': 'LSTM', 'units': n_features * lookback, 'return_sequences': True},  
            {'type': 'LSTM', 'units': n_features * lookback, 'return_sequences': False},      
            {'type': 'Dense', 'units': 1, 'activation': 'selu'}
        ]

        mlflow.log_param("model_layers_config", model_layers_config)

        model = create_lstm_model(model_layers_config)
        model.compile(optimizer=Adam(learning_rate=params['learning_rate']), loss=params['loss'], metrics=params['metrics'])

        # prep data
        X, y = create_sequences_np(data=training, features=features, lookback=lookback, datatype=params['datatype'])

        training_validation_split = 0.8
        split_idx = int(len(X) * training_validation_split)

        mlflow.log_param("training_validation_split", training_validation_split)

        # validation set with the last 20 percent of the dataset 
        X_train, X_val = X[:split_idx], X[split_idx:]
        y_train, y_val = y[:split_idx], y[split_idx:]

        mlflow.log_param("X_train.shape", X_train.shape)
        mlflow.log_param("y_train.shape", y_train.shape)

        mlflow.log_param("X_val.shape", X_val.shape)
        mlflow.log_param("y_val.shape", y_val.shape)
        
        # train model
        model.fit(X_train, y_train, epochs=params['epochs'], batch_size=params['batch_size'], validation_data=(X_val, y_val), callbacks=[early_stopping]) # , callbacks=[early_stopping]
        
        # Evaluation
        # Validation Data
        y_pred = model.predict(X_val)

        y_val_true = np.expm1(y_val)
        y_pred_true = np.expm1(y_pred)

        fig, ax = plt.subplots(figsize=(15, 7))
        ax.plot(y_val_true, label='Actual Values')
        ax.plot(y_pred_true, label='Predicted Values')
        ax.set_title('LSTM Model Validation Plot')
        ax.set_xlabel('Time')
        ax.set_ylabel('Close Price')
        ax.set_yscale('log')
        ax.legend()
        fig.tight_layout()
        artifact_path = 'plots'
        mlflow.log_figure(fig, artifact_path + '/validation_plot.png')
        plt.close(fig)

        # Unseen Data
        predicted_prices = []
        actual_prices = test['close_price_true'].values[lookback:]  # actual prices without log transformation
        timestamps = test['bucket'].values[lookback:]  # corresponding timestamps

        for i in range(lookback, len(test)):
            last_sequence = test.iloc[i-lookback:i][features].values.reshape((1, lookback, len(features)))
            last_sequence = np.array(last_sequence).astype(np.float16)
            predicted_log_price = model.predict(last_sequence)
            predicted_price = np.expm1(predicted_log_price)[0, 0]  # inverse log transformation
            predicted_prices.append(predicted_price)
        predicted_prices = np.array(predicted_prices)

        # create and log plot using mlflow
        fig, ax = plt.subplots(figsize=(15, 7))
        ax.plot(timestamps, actual_prices, label='Actual Prices', color='blue')
        ax.plot(timestamps, predicted_prices, label='Predicted Prices', color='orange')
        ax.set_title('LSTM Model Predictions vs Actual Prices')
        ax.set_xlabel('Date/Time')
        ax.set_ylabel('Close Price')
        labels = ax.get_xticklabels()
        ax.legend()
        fig.tight_layout()
        artifact_path = 'plots'
        mlflow.log_figure(fig, artifact_path + '/unseen_test_plot.png')
        plt.close(fig)

        # calculate some metrics 
        mse_value = mean_squared_error(actual_prices, predicted_prices)
        mae_value = mean_absolute_error(actual_prices, predicted_prices)
        rmse_value = np.sqrt(mse_value)
        mape_value = np.mean(np.abs((actual_prices - predicted_prices) / actual_prices)) * 100
        r2_value = r2_score(actual_prices, predicted_prices)
        explained_variance = explained_variance_score(actual_prices, predicted_prices)

        # log metrics
        mlflow.log_metric('mse', mse_value)
        mlflow.log_metric('mae', mae_value)
        mlflow.log_metric('rmse', rmse_value)
        mlflow.log_metric('mape', mape_value)
        mlflow.log_metric('r2', r2_value)
        mlflow.log_metric('explained_variance', explained_variance)

        mlflow.sklearn.log_model(model, "model")

        run_count = run_count + 1


0


2024-02-07 16:54:56.677384: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-02-07 16:54:56.677618: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-02-07 16:54:56.677793: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

Epoch 1/75


2024-02-07 16:54:59.315090: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 24843840 exceeds 10% of free system memory.
2024-02-07 16:54:59.363089: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 24843840 exceeds 10% of free system memory.
2024-02-07 16:55:05.255760: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8700


   1/1618 [..............................] - ETA: 3:28:33 - loss: 104.2500 - mean_absolute_error: 7.5072 - mean_squared_error: 57.4281 - accuracy: 0.0000e+00

2024-02-07 16:55:07.170635: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7f169cbfa1c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-02-07 16:55:07.170662: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce GTX 1060 6GB, Compute Capability 6.1
2024-02-07 16:55:07.189891: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-02-07 16:55:07.281868: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1618/1618 [==============================] - 22s 9ms/step - loss: 2.6187 - mean_absolute_error: 0.1780 - mean_squared_error: 0.3087 - accuracy: 0.0000e+00 - val_loss: 0.5145 - val_mean_absolute_error: 0.0383 - val_mean_squared_error: 0.0022 - val_accuracy: 0.0000e+00
Epoch 2/75
1618/1618 [==============================] - 12s 7ms/step - loss: 0.8943 - mean_absolute_error: 0.0613 - mean_squared_error: 0.0061 - accuracy: 0.0000e+00 - val_loss: 1.3888 - val_mean_absolute_error: 0.1034 - val_mean_squared_error: 0.0115 - val_accuracy: 0.0000e+00
Epoch 3/75
1618/1618 [==============================] - 12s 7ms/step - loss: 0.7820 - mean_absolute_error: 0.0536 - mean_squared_error: 0.0048 - accuracy: 0.0000e+00 - val_loss: 0.4030 - val_mean_absolute_error: 0.0301 - val_mean_squared_error: 0.0013 - val_accuracy: 0.0000e+00
Epoch 4/75
1618/1618 [==============================] - 12s 8ms/step - loss: 0.7711 - mean_absolute_error: 0.0528 - mean_squared_error: 0.0045 - accuracy: 0.0000e+00 - val_lo

2024/02/07 17:05:09 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpnwo9dab2/model/model.pkl, flavor: sklearn), fall back to return ['scikit-learn==1.1.3', 'cloudpickle==2.2.1']. Set logging level to DEBUG to see the full traceback.
/home/rileydavid/.local/lib/python3.10/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
